# 01: System Diagnostics, Hardware Acceleration & Environment Setup

**Track 01: Foundations, Math & Diagnostics** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Verify hardware accelerators (NVIDIA CUDA, Apple Silicon MPS/Metal, or CPU AVX-512), validate library compatibility across PyTorch, SciPy, Scikit-Learn, and inspect multi-threading capabilities.


## 1. Environment & Architecture Inspection
Let's check Python version, operating system environment, memory allocation, and CPU count.

In [ ]:
import sys
import os
import platform
import psutil
import torch

print("=== System Architecture Diagnostics ===")
print(f"Python Version    : {sys.version.split()[0]}")
print(f"Platform          : {platform.system()} {platform.release()} ({platform.machine()})")
print(f"CPU Physical Cores: {psutil.cpu_count(logical=False)}")
print(f"CPU Logical Cores : {psutil.cpu_count(logical=True)}")
print(f"Total RAM (GB)    : {psutil.virtual_memory().total / (1024**3):.2f} GB")
print(f"Available RAM (GB): {psutil.virtual_memory().available / (1024**3):.2f} GB")

## 2. Hardware Acceleration Diagnostics (CUDA vs MPS vs CPU)
Tensorbox seamlessly handles CUDA on Linux/Windows and Apple Silicon Metal (MPS) on macOS.

In [ ]:
def detect_compute_device():
    if torch.cuda.is_available():
        device_name = torch.cuda.get_device_name(0)
        device = torch.device("cuda")
        print(f"✓ NVIDIA CUDA Acceleration Active: {device_name}")
        print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
        print("✓ Apple Silicon Metal Performance Shaders (MPS) Active")
    else:
        device = torch.device("cpu")
        print("✓ CPU Compute Engine Active (AVX/SIMD Acceleration)")
    return device

device = detect_compute_device()
print(f"Primary Torch Device: {device}")

## 3. Matrix Multiplication Benchmark on Active Device
Benchmarking a large $2000 \times 2000$ matrix multiplication to confirm hardware acceleration throughput.

In [ ]:
import time

size = 2000
A = torch.randn(size, size, device=device)
B = torch.randn(size, size, device=device)

# Warmup
_ = torch.matmul(A, B)
if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()
for _ in range(20):
    C = torch.matmul(A, B)
if device.type == "cuda":
    torch.cuda.synchronize()
elapsed = (time.perf_counter() - start_time) / 20

print(f"Mean execution time for {size}x{size} MatMul on [{device}]: {elapsed*1000:.2f} ms")
print(f"Result matrix shape: {C.shape}, Mean value: {C.mean().item():.4f}")

## 4. Diagnostic Summary
- All core numerical backends and PyTorch compute engines are validated.
- Environment is ready for deep learning, tabular machine learning, and high-performance data processing.